# Notebook for extracting mandatory subjects for all schools

In [40]:
import json
import os
import sys
import pandas as pd
from firecrawl_app import app, ExtractSchema, NestedModel
from utils.helpers import *
from pydantic import BaseModel


sys.path.append(os.path.abspath(".."))

# Load config
filepath = "config_mandatory.json"
with open(filepath, "r", encoding="utf-8") as json_file:
    config_data = json.load(json_file)

# Print schools 
schools = list(config_data.keys())
print("Schools currently defined in config_mandatory.json:")
print("-" * 40)
for school in sorted(schools):
    print(f"- {school}")


Schools currently defined in config_mandatory.json:
----------------------------------------
- HIOF
- KRISTIANIA
- NTNU
- UIA
- UIB
- UIO
- UIS
- UIT
- USN


In [41]:
# Choose school
school = "UIA"

## Code for running extraction of mandatory subjects for all schools defined in config_mandatory.json.

In [42]:
school_data = config_data[school]

sys.path.append(os.path.abspath(".."))

class DummySchema(BaseModel):
    obligatorise_emner: list[str]

def try_single_prompt(url, prompt):
    try:
        data = app.extract([url], {
            'prompt': prompt,
            'schema': DummySchema.model_json_schema()
        })
        return data.get("data", {}).get("obligatorise_emner", [])
    except Exception as e:
        print(f"Prompt failed: {prompt[:50]}...\nError: {e}")
        return []


print(f" Extracting mandatory subjects for {school}...\n")
rows = []
def try_multiple_prompts(url, prompts, app):
    for prompt in prompts:
        try:
            data = app.extract([url], {
                'prompt': prompt,
                'schema': ExtractSchema.model_json_schema()
            })
            subjects = data["data"]["læringsutbyttebeskrivelser"].get("obligatorise_emner", [])
            if subjects:
                return subjects, prompt
        except Exception as e:
            print(f"Prompt failed: {prompt}\nError: {e}")
    return [], None


for study_program, url in school_data["mandatory_subjects"].items():
    print(f" - Extracting for program: {study_program}")
    
    try:
        prompts = school_data["prompt_mandatory_subjects"]
        if isinstance(prompts, str):
            prompts = [prompts]  # Make sure it's a list

        subjects, used_prompt = try_multiple_prompts(url, prompts, app)

        if subjects:
            for subject in subjects:
                rows.append({
                    "Skole": school,
                    "Studie program": study_program,
                    "Læringsutbytte type": "Obligatorise_emner",
                    "Læringsutbytte": subject.strip()
                })
        else:
            print("    No mandatory subjects found.")


    except Exception as e:
        print(f"  Failed to extract for {study_program}: {e}")

# Write using helper functions
if rows:
    df = pd.DataFrame(rows)
    filename = f"{school}_Mandatory_subjects.csv"
    df.to_csv(filename, index=False)
    print(f"\nData saved to {filename}")

    csv_to_excel(filename)

else:
    print("\nNo data extracted.")


 Extracting mandatory subjects for UIA...

 - Extracting for program: Bachelor i ingenioerfag, data

Data saved to UIA_Mandatory_subjects.csv

Excel file saved: UIA_Mandatory_subjects.xlsx



📘 Program: Bachelor i ingenioerfag, data
Prompt length: 236 — Found 22 subjects
Prompt length: 115 — Found 31 subjects
Prompt length: 83 — Found 35 subjects
Prompt length: 114 — Found 34 subjects

🔁 Super prompt found 32 subjects

✅ Saved: UIA_BEST_PROMPT_comparison.csv
